# Behavior Classification MLP — Training Notebook

**Project:** AI-Based Person Detection and Behavior Recognition System (FYP2)  
**Model:** BehaviorMLP — 4-layer MLP on 17-keypoint pose sequences  
**Target:** ≥80% validation accuracy on 7 behavior classes  
**Hardware:** NVIDIA RTX 5060 (8GB VRAM) — FP16 training

## Behavior Classes
| Index | Class | NTU RGB+D Action IDs |
|-------|-------|----------------------|
| 0 | walking | A008, A009, A010 |
| 1 | standing | A012 |
| 2 | sitting | A013 |
| 3 | running | A011 |
| 4 | bending | A017, A024 |
| 5 | falling | A043, A044, A045 |
| 6 | suspicious | A068, A069 (loitering/lingering) |

## Pipeline
```
NTU RGB+D 120 skeleton data (.pkl)
  → Filter to 7 target action classes
  → Convert 25-joint NTU → 17-joint COCO mapping
  → Normalize keypoints (bbox-relative)
  → Sliding window (15 frames → 765 features)
  → Train BehaviorMLP
  → Save best model to ../models/behavior_mlp.pth
```

## Dataset Setup (run once)
```bash
# Option A — NTU RGB+D 120 2D skeleton (MMAction2 format, 17 COCO keypoints, ~200MB)
# Register at: https://rose1.ntu.edu.sg/dataset/actionRecognition/
# Then download the MMAction2 pickle (already converted to COCO format):
# wget https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu120_2d.pkl

# Option B — NTU RGB+D 60 2D skeleton (smaller, 60 classes, faster to start with)
# wget https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl

# Place the file at:
# backend/data/ntu120_2d.pkl  (or ntu60_2d.pkl)
```


## 0. Configuration — Change These Before Running

In [ ]:
# ============================================================
# CONFIGURATION — adjust these as needed
# ============================================================

# --- Data ---
DATA_PKL = "../data/ntu120_2d.pkl"        # path to NTU RGB+D pkl file
# DATA_PKL = "../data/ntu60_2d.pkl"        # smaller alternative

# --- Model Output ---
SAVE_DIR = "../models"
MODEL_NAME = "behavior_mlp_v1"             # checkpoint prefix

# --- Architecture ---
POSE_WINDOW_SIZE = 15                      # frames per sample (must match pipeline)
NUM_KEYPOINTS    = 17                      # COCO skeleton keypoints
FEATURES_PER_KP  = 3                       # x, y, confidence
INPUT_SIZE       = NUM_KEYPOINTS * FEATURES_PER_KP * POSE_WINDOW_SIZE  # 765
NUM_CLASSES      = 7
HIDDEN_SIZES     = [256, 128, 64]          # experiment: try [512,256,128] for more capacity
DROPOUT          = 0.3

# --- Training ---
BATCH_SIZE       = 256
EPOCHS           = 80
LEARNING_RATE    = 1e-3
WEIGHT_DECAY     = 1e-4
PATIENCE         = 15                      # early stopping patience
VAL_SPLIT        = 0.15
TEST_SPLIT       = 0.10
RANDOM_SEED      = 42
NUM_WORKERS      = 4
USE_AMP          = True                    # FP16 mixed-precision (RTX 5060)

# --- Augmentation ---
AUG_NOISE_STD    = 0.01                    # Gaussian noise on keypoints
AUG_SCALE_RANGE  = (0.9, 1.1)             # random bbox scale jitter
AUG_FLIP_PROB    = 0.5                     # horizontal flip probability

# ============================================================
BEHAVIOR_CLASSES = ["walking", "standing", "sitting", "running", "bending", "falling", "suspicious"]

# NTU RGB+D action IDs → behavior label mapping
# NTU uses 1-indexed action IDs (A001 = 1)
NTU_CLASS_MAP = {
    # walking variants
    8:  "walking",   # walk
    9:  "walking",   # walk toward each other
    10: "walking",   # push
    # standing
    12: "standing",  # stand up
    # sitting
    13: "sitting",   # sit down
    # running
    11: "running",   # run
    # bending
    17: "bending",   # bend over
    24: "bending",   # pick up
    # falling
    43: "falling",   # falling
    44: "falling",   # headache (slumping)
    45: "falling",   # chest pain (collapsing)
    # suspicious (loitering/lurking)
    68: "suspicious", # cross arms
    69: "suspicious", # look around
    # extra classes to improve balance
    3:  "standing",  # stand up from lying
    7:  "sitting",   # sit down on chair
    20: "bending",   # drop
    55: "walking",   # squat down (transition)
}

print(f"Input size: {INPUT_SIZE} features")
print(f"Classes: {BEHAVIOR_CLASSES}")
print(f"NTU actions mapped: {len(NTU_CLASS_MAP)}")

## 1. Imports & Setup

In [ ]:
import os
import sys
import pickle
import random
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    top_k_accuracy_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 2. Load & Parse NTU RGB+D Dataset

In [ ]:
def load_ntu_pkl(pkl_path: str) -> dict:
    """Load the MMAction2-format NTU RGB+D skeleton pickle."""
    print(f"Loading {pkl_path} ...")
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    print(f"  Keys: {list(data.keys())}")
    
    # MMAction2 format: split_name, annotations list
    # annotations = [{keypoint, keypoint_score, label, frame_dir, total_frames, ...}]
    annots = data.get("annotations", data.get("split", {}))
    
    # If it's split dict (train/val keys)
    if isinstance(annots, dict):
        # Merge train + val sample names
        all_samples = []
        for split_name, sample_list in annots.items():
            all_samples.extend(sample_list)
        return {"annotations": all_samples}
    
    return data


def extract_sample_features(
    keypoints: np.ndarray,
    keypoint_scores: np.ndarray,
    window_size: int = 15
) -> np.ndarray:
    """
    Convert a variable-length pose sequence to fixed-size feature vector.
    
    Args:
        keypoints: (T, M, V, C) — T=frames, M=persons, V=17 joints, C=2 (x,y)
        keypoint_scores: (T, M, V) — per-keypoint confidence
        window_size: number of frames to use
    
    Returns:
        features: (window_size * 17 * 3,) = 765-dim vector
    """
    T, M, V, C = keypoints.shape  # e.g. (100, 1, 17, 2)
    
    # Use only person 0 (primary subject)
    kp  = keypoints[:, 0, :, :]      # (T, 17, 2)
    sc  = keypoint_scores[:, 0, :]   # (T, 17)
    
    # Resample to window_size frames uniformly
    indices = np.linspace(0, T - 1, window_size).astype(int)
    kp_sampled = kp[indices]   # (window_size, 17, 2)
    sc_sampled = sc[indices]   # (window_size, 17)
    
    # Normalize keypoints relative to bounding box
    frames_out = []
    for t in range(window_size):
        xy   = kp_sampled[t]    # (17, 2)
        conf = sc_sampled[t]    # (17,)
        
        x_vals = xy[:, 0]
        y_vals = xy[:, 1]
        x_min, x_max = x_vals.min(), x_vals.max()
        y_min, y_max = y_vals.min(), y_vals.max()
        w = max(x_max - x_min, 1e-5)
        h = max(y_max - y_min, 1e-5)
        
        x_norm = (x_vals - x_min) / w
        y_norm = (y_vals - y_min) / h
        
        # Stack: [x0,y0,c0, x1,y1,c1, ...] = 51 values per frame
        frame_feat = np.stack([x_norm, y_norm, conf], axis=1).flatten()  # (51,)
        frames_out.append(frame_feat)
    
    return np.concatenate(frames_out).astype(np.float32)  # (765,)


print("Functions defined. Now loading data...")

In [ ]:
# ── Load pickle ──────────────────────────────────────────────────────────────
if not Path(DATA_PKL).exists():
    print(f"[ERROR] Dataset not found at: {DATA_PKL}")
    print("Please download and place the file. See instructions in cell 0.")
    print("")
    print("QUICK TEST: generating synthetic data to verify the pipeline works...")
    
    # Generate synthetic data so the rest of the notebook can run
    N_SYNTH = 5000
    np.random.seed(42)
    X_raw  = np.random.randn(N_SYNTH, INPUT_SIZE).astype(np.float32)
    y_raw  = np.random.randint(0, NUM_CLASSES, N_SYNTH)
    print(f"  Synthetic samples: {N_SYNTH}")
    USING_SYNTHETIC = True
else:
    data = load_ntu_pkl(DATA_PKL)
    annotations = data.get("annotations", [])
    print(f"  Total samples in file: {len(annotations)}")
    USING_SYNTHETIC = False
    
    # ── Filter to target classes ─────────────────────────────────────────────
    X_list, y_list = [], []
    skipped = 0
    
    for sample in annotations:
        # NTU action label is 1-indexed in the label field
        raw_label = sample.get("label", -1)
        
        if raw_label not in NTU_CLASS_MAP:
            skipped += 1
            continue
        
        behavior = NTU_CLASS_MAP[raw_label]
        label_idx = BEHAVIOR_CLASSES.index(behavior)
        
        kp     = sample["keypoint"]         # (T, M, V, 2)
        kp_sc  = sample["keypoint_score"]   # (T, M, V)
        
        # Skip if too short
        if kp.shape[0] < 5:
            skipped += 1
            continue
        
        feats = extract_sample_features(kp, kp_sc, POSE_WINDOW_SIZE)
        X_list.append(feats)
        y_list.append(label_idx)
    
    X_raw = np.array(X_list, dtype=np.float32)
    y_raw = np.array(y_list, dtype=np.int64)
    print(f"  Skipped (not in target classes): {skipped}")
    print(f"  Used: {len(X_raw)} samples")

# ── Class distribution ───────────────────────────────────────────────────────
counts = Counter(y_raw.tolist())
print("\nClass distribution:")
for idx, name in enumerate(BEHAVIOR_CLASSES):
    n = counts.get(idx, 0)
    bar = "█" * (n // max(max(counts.values()) // 40, 1))
    print(f"  [{idx}] {name:12s}  {n:5d}  {bar}")

## 3. Data Augmentation

In [ ]:
def augment_keypoints(features: np.ndarray) -> np.ndarray:
    """
    Apply pose augmentation to a feature vector (765,).
    Features are [x0,y0,c0, x1,y1,c1, ...] * 15 frames.
    Only augments x,y — not confidence scores.
    """
    feat = features.copy().reshape(POSE_WINDOW_SIZE, NUM_KEYPOINTS, FEATURES_PER_KP)
    
    # 1. Gaussian noise
    feat[:, :, :2] += np.random.normal(0, AUG_NOISE_STD, feat[:, :, :2].shape)
    
    # 2. Horizontal flip (mirror x coords: x → 1-x)
    if np.random.random() < AUG_FLIP_PROB:
        feat[:, :, 0] = 1.0 - feat[:, :, 0]
        # Swap left/right keypoints (COCO: 1↔2, 3↔4, 5↔6, 7↔8, 9↔10, 11↔12, 13↔14, 15↔16)
        swap_pairs = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
        for a, b in swap_pairs:
            feat[:, a, :], feat[:, b, :] = feat[:, b, :].copy(), feat[:, a, :].copy()
    
    # 3. Random scale (zoom)
    scale = np.random.uniform(*AUG_SCALE_RANGE)
    feat[:, :, :2] *= scale
    feat[:, :, :2] = np.clip(feat[:, :, :2], 0, 1)
    
    # 4. Temporal jitter — randomly drop 1-2 frames and repeat neighbours
    if np.random.random() < 0.3:
        drop_idx = np.random.randint(1, POSE_WINDOW_SIZE - 1)
        feat[drop_idx] = feat[drop_idx - 1]
    
    return feat.reshape(-1).astype(np.float32)


class BehaviorDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y.astype(np.int64))
        self.augment = augment
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].numpy()
        if self.augment:
            x = augment_keypoints(x)
            x = torch.from_numpy(x)
        else:
            x = self.X[idx]
        return x, self.y[idx]


# ── Train/Val/Test split ─────────────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_raw, test_size=TEST_SPLIT, stratify=y_raw, random_state=RANDOM_SEED
)
val_frac = VAL_SPLIT / (1 - TEST_SPLIT)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_frac, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

# ── Balanced sampling (handles class imbalance) ───────────────────────────────
class_counts = np.bincount(y_train)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# ── DataLoaders ──────────────────────────────────────────────────────────────
train_ds = BehaviorDataset(X_train, y_train, augment=True)
val_ds   = BehaviorDataset(X_val,   y_val,   augment=False)
test_ds  = BehaviorDataset(X_test,  y_test,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

## 4. Model Architecture

In [ ]:
class BehaviorMLP(nn.Module):
    """
    4-layer MLP for behavior classification from pose keypoint sequences.
    Matches the architecture used in app/pipeline/classifier.py exactly.
    
    Input:  (batch, 765)  — 17 keypoints × 3 values × 15 frames
    Output: (batch, 7)    — logits for 7 behavior classes
    """
    def __init__(
        self,
        input_size:  int   = INPUT_SIZE,
        hidden_sizes       = HIDDEN_SIZES,
        num_classes: int   = NUM_CLASSES,
        dropout:     float = DROPOUT
    ):
        super().__init__()
        self.input_size  = input_size
        self.num_classes = num_classes
        
        layers = []
        in_dim = input_size
        
        for i, out_dim in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.BatchNorm1d(out_dim))
            layers.append(nn.ReLU())
            # Last hidden layer uses half dropout
            drop = dropout if i < len(hidden_sizes) - 1 else dropout * 0.5
            layers.append(nn.Dropout(drop))
            in_dim = out_dim
        
        layers.append(nn.Linear(in_dim, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


model = BehaviorMLP().to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: BehaviorMLP")
print(f"  Input  → {HIDDEN_SIZES[0]} → {HIDDEN_SIZES[1]} → {HIDDEN_SIZES[2]} → {NUM_CLASSES}")
print(f"  Total params: {total_params:,}")
print(f"  Trainable:    {trainable:,}")
print(model)

## 5. Training Loop

In [ ]:
# ── Loss, Optimizer, Scheduler ───────────────────────────────────────────────
# Class-weighted cross-entropy to handle imbalanced classes
class_weight_tensor = torch.tensor(
    class_weights / class_weights.sum() * NUM_CLASSES,
    dtype=torch.float32
).to(DEVICE)

criterion  = nn.CrossEntropyLoss(weight=class_weight_tensor)
optimizer  = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler     = GradScaler(enabled=USE_AMP and DEVICE.type == "cuda")


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        
        with autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
        
        if train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            scaler.step(optimizer)
            scaler.update()
        
        total_loss += loss.item() * len(y_batch)
        preds       = logits.argmax(dim=1)
        correct    += (preds == y_batch).sum().item()
        total      += len(y_batch)
    
    return total_loss / total, correct / total


# ── Training loop with early stopping ────────────────────────────────────────
best_val_acc = 0.0
best_epoch   = 0
patience_ctr = 0
history      = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

best_path = Path(SAVE_DIR) / f"{MODEL_NAME}_best.pth"

print(f"Training for up to {EPOCHS} epochs (early stop patience={PATIENCE})")
print(f"{'Epoch':>6} {'LR':>8} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'Best':>5}")
print("-" * 66)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    scheduler.step()
    
    lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(lr)
    
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_epoch   = epoch
        patience_ctr = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "train_acc": train_acc,
            "config": {
                "input_size": INPUT_SIZE, "hidden_sizes": HIDDEN_SIZES,
                "num_classes": NUM_CLASSES, "dropout": DROPOUT,
                "behavior_classes": BEHAVIOR_CLASSES,
                "pose_window_size": POSE_WINDOW_SIZE,
            }
        }, best_path)
    else:
        patience_ctr += 1
    
    star = "★" if is_best else " "
    target = " ✓" if val_acc >= 0.80 else ""
    print(f"{epoch:>6} {lr:>8.2e} {train_loss:>11.4f} {train_acc:>9.1%} {val_loss:>10.4f} {val_acc:>8.1%}{target} {star}")
    
    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

print(f"\nBest val accuracy: {best_val_acc:.1%} at epoch {best_epoch}")
print(f"Model saved: {best_path}")

## 6. Training Curves

In [ ]:
epochs_ran = len(history["train_loss"])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(x, history["train_loss"], label="Train", color="steelblue")
axes[0].plot(x, history["val_loss"],   label="Val",   color="tomato")
axes[0].axvline(best_epoch, color="gold", ls="--", lw=1.5, label=f"Best (e{best_epoch})")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

# Accuracy
axes[1].plot(x, history["train_acc"], label="Train", color="steelblue")
axes[1].plot(x, history["val_acc"],   label="Val",   color="tomato")
axes[1].axhline(0.80, color="green", ls=":", lw=1.5, label="80% target")
axes[1].axvline(best_epoch, color="gold", ls="--", lw=1.5, label=f"Best (e{best_epoch})")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# Learning rate
axes[2].plot(x, history["lr"], color="purple")
axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

plt.suptitle(f"BehaviorMLP Training — Best val acc: {best_val_acc:.1%} (epoch {best_epoch})", y=1.02)
plt.tight_layout()
plt.savefig(Path(SAVE_DIR) / f"{MODEL_NAME}_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved training curves.")

## 7. Evaluation on Test Set

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# ── Collect predictions ───────────────────────────────────────────────────────
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits  = model(X_batch)
        probs   = torch.softmax(logits, dim=1).cpu().numpy()
        preds   = logits.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())
        all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

test_acc = accuracy_score(all_labels, all_preds)
top2_acc = top_k_accuracy_score(all_labels, all_probs, k=2) if NUM_CLASSES > 2 else None

print(f"Test Accuracy:    {test_acc:.1%}")
if top2_acc:
    print(f"Top-2 Accuracy:   {top2_acc:.1%}")
print(f"\nTarget ≥80%: {'✓ PASSED' if test_acc >= 0.80 else '✗ NOT YET — experiment further'}")
print()
print(classification_report(all_labels, all_preds, target_names=BEHAVIOR_CLASSES))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues",
            xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES, ax=axes[1])
axes[1].set_title("Confusion Matrix (normalized)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig(Path(SAVE_DIR) / f"{MODEL_NAME}_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Export Model for Pipeline

In [ ]:
# ── Save final model in pipeline-compatible format ────────────────────────────
final_path = Path(SAVE_DIR) / f"{MODEL_NAME}_final.pth"

torch.save(model.state_dict(), final_path)

# Save config separately so pipeline can verify compatibility
config_path = Path(SAVE_DIR) / f"{MODEL_NAME}_config.json"
model_config = {
    "input_size":         INPUT_SIZE,
    "hidden_sizes":       HIDDEN_SIZES,
    "num_classes":        NUM_CLASSES,
    "dropout":            DROPOUT,
    "behavior_classes":   BEHAVIOR_CLASSES,
    "pose_window_size":   POSE_WINDOW_SIZE,
    "test_accuracy":      float(test_acc),
    "best_val_accuracy":  float(best_val_acc),
    "trained_on":         "NTU RGB+D 120" if not USING_SYNTHETIC else "SYNTHETIC",
    "ntu_class_map":      {str(k): v for k, v in NTU_CLASS_MAP.items()},
}
with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)

print(f"Model state dict → {final_path}")
print(f"Config           → {config_path}")
print()
print("To use in pipeline, set in your .env:")
print(f'  MLP_MODEL_PATH = "models/{MODEL_NAME}_final.pth"')

## 9. Experiment Variants (change and rerun cells 4–8)

The table below tracks experiments. Update manually after each run.

| # | Hidden Sizes | Dropout | LR | Augmentation | Val Acc | Test Acc | Notes |
|---|---|---|---|---|---|---|---|
| baseline | [256,128,64] | 0.3 | 1e-3 | yes | — | — | starting point |
| exp1 | [512,256,128] | 0.3 | 1e-3 | yes | — | — | wider |
| exp2 | [256,128,64] | 0.4 | 5e-4 | yes | — | — | more dropout |
| exp3 | [256,128,64] | 0.3 | 1e-3 | no  | — | — | no augmentation |
| exp4 | [512,256,128,64] | 0.3 | 1e-3 | yes | — | — | deeper |

### To run an experiment:
1. Change `HIDDEN_SIZES`, `DROPOUT`, `LEARNING_RATE`, or augmentation params in cell 0
2. Change `MODEL_NAME = "behavior_mlp_v2"` (or v3, etc.) to keep checkpoints separate
3. Restart kernel and run all cells
4. Record results in the table above

### Ideas to reach 80% if baseline falls short:
- Add **velocity features**: for each keypoint, compute `kp[t] - kp[t-1]` and concatenate → 1530 features
- Use **LSTM** instead of MLP (temporal dependencies, try cell below)
- Add **more NTU classes** to the `NTU_CLASS_MAP`
- Mix in CAVIAR skeleton data for real CCTV footage distribution
- Apply **label smoothing** in CrossEntropyLoss (`label_smoothing=0.1`)

In [ ]:
# ── OPTIONAL: LSTM alternative architecture ───────────────────────────────────
# Uncomment this cell and adapt cells 4-8 to use BehaviorLSTM instead of BehaviorMLP
# The input needs reshaping: (batch, 765) → (batch, 15, 51)

class BehaviorLSTM(nn.Module):
    """
    LSTM-based behavior classifier.
    Better at capturing temporal patterns (walking rhythm, fall trajectory).
    Slightly slower than MLP at inference.
    
    Input:  (batch, 765) — will be reshaped to (batch, 15, 51)
    Output: (batch, 7)
    """
    def __init__(self, input_size=INPUT_SIZE, num_classes=NUM_CLASSES,
                 hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.seq_len   = POSE_WINDOW_SIZE           # 15
        self.feat_dim  = input_size // POSE_WINDOW_SIZE  # 51
        self.lstm = nn.LSTM(
            input_size=self.feat_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        x = x.view(x.size(0), self.seq_len, self.feat_dim)  # (B, 15, 51)
        out, _ = self.lstm(x)                                # (B, 15, 256)
        out    = out[:, -1, :]                               # last frame: (B, 256)
        return self.head(out)


# Test LSTM shape
lstm_test = BehaviorLSTM().to(DEVICE)
dummy = torch.randn(4, INPUT_SIZE).to(DEVICE)
out   = lstm_test(dummy)
print(f"LSTM output shape: {out.shape}  (expected: [4, {NUM_CLASSES}])")
lstm_params = sum(p.numel() for p in lstm_test.parameters())
print(f"LSTM params: {lstm_params:,}")

## 10. Dataset Health Check

In [ ]:
# ── Visual check of class balance ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full dataset
counts_all = [Counter(y_raw.tolist()).get(i, 0) for i in range(NUM_CLASSES)]
axes[0].bar(BEHAVIOR_CLASSES, counts_all, color="steelblue", edgecolor="white")
axes[0].set_title("Full Dataset Distribution")
axes[0].set_xlabel("Behavior")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts_all):
    axes[0].text(i, v + 5, str(v), ha="center", fontsize=9)

# Train split only
counts_train = [Counter(y_train.tolist()).get(i, 0) for i in range(NUM_CLASSES)]
axes[1].bar(BEHAVIOR_CLASSES, counts_train, color="tomato", edgecolor="white")
axes[1].set_title("Train Split Distribution (before sampling)")
axes[1].set_xlabel("Behavior")
for i, v in enumerate(counts_train):
    axes[1].text(i, v + 5, str(v), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(Path(SAVE_DIR) / f"{MODEL_NAME}_class_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nIMBALANCE WARNING" if max(counts_all) > 3 * min(c for c in counts_all if c > 0) else "\nClass balance looks acceptable.")